In [1]:
def bayes(prior_hazard, observation):
    p_alarm_given_hazard = 0.90
    p_alarm_given_safe = 0.10

    if observation == "ALARM":
        p_obs_given_hazard = p_alarm_given_hazard
        p_obs_given_safe = p_alarm_given_safe
    elif observation == "NORMAL":
        p_obs_given_hazard = 1 - p_alarm_given_hazard
        p_obs_given_safe = 1 - p_alarm_given_safe
    else:
        raise ValueError("Unknown observation")

    prior_safe = 1 - prior_hazard

    numerator = prior_hazard * p_obs_given_hazard
    denominator = numerator + prior_safe * p_obs_given_safe

    posterior_hazard = numerator / denominator
    posterior_safe = 1 - posterior_hazard

    return posterior_hazard, posterior_safe


print("ALARM:", bayes(0.20, "ALARM"))
print("NORMAL:", bayes(0.20, "NORMAL"))

ALARM: (0.6923076923076924, 0.3076923076923076)
NORMAL: (0.027027027027027018, 0.972972972972973)


In [2]:
priors = [0.10, 0.20, 0.30]

print("Sensitivity analysis for ALARM:")

for prior in priors:
    hazard, safe = bayes(prior, "ALARM")
    print(
        f"Prior = {prior:.2f}, "
        f"P(HAZARD | ALARM) = {hazard:.4f}, "
        f"P(SAFE | ALARM) = {safe:.4f}"
    )

Sensitivity analysis for ALARM:
Prior = 0.10, P(HAZARD | ALARM) = 0.5000, P(SAFE | ALARM) = 0.5000
Prior = 0.20, P(HAZARD | ALARM) = 0.6923, P(SAFE | ALARM) = 0.3077
Prior = 0.30, P(HAZARD | ALARM) = 0.7941, P(SAFE | ALARM) = 0.2059


In [3]:
def hmm_filter(observations):
    # Initial probabilities
    p_hazard = 0.20
    p_safe = 0.80

    # Transition probabilities
    p_hazard_to_hazard = 0.80
    p_safe_to_hazard = 0.10
    p_hazard_to_safe = 0.20
    p_safe_to_safe = 0.90

    # Observation probabilities
    p_alarm_given_hazard = 0.90
    p_alarm_given_safe = 0.10

    results = []

    for t, observation in enumerate(observations):
        # Prediction
        if t == 0:
            predicted_hazard = p_hazard
            predicted_safe = p_safe
        else:
            predicted_hazard = (
                p_hazard * p_hazard_to_hazard
                + p_safe * p_safe_to_hazard
            )

            predicted_safe = (
                p_hazard * p_hazard_to_safe
                + p_safe * p_safe_to_safe
            )

        # Observation update
        if observation == "ALARM":
            likelihood_hazard = p_alarm_given_hazard
            likelihood_safe = p_alarm_given_safe
        elif observation == "NORMAL":
            likelihood_hazard = 1 - p_alarm_given_hazard
            likelihood_safe = 1 - p_alarm_given_safe
        else:
            raise ValueError("Unknown observation")

        unnormalized_hazard = predicted_hazard * likelihood_hazard
        unnormalized_safe = predicted_safe * likelihood_safe

        # Normalization
        total = unnormalized_hazard + unnormalized_safe

        p_hazard = unnormalized_hazard / total
        p_safe = unnormalized_safe / total

        results.append((
            t + 1,
            observation,
            predicted_hazard,
            p_hazard,
            predicted_safe,
            p_safe
        ))

    return results


observations = ["ALARM", "NORMAL", "ALARM", "ALARM"]

results = hmm_filter(observations)

print("Time | Observation | Predicted H | Updated H | Predicted S | Updated S")

for row in results:
    t, obs, pred_h, upd_h, pred_s, upd_s = row
    print(
        f"{t:>4} | {obs:>11} | "
        f"{pred_h:.4f} | {upd_h:.4f} | "
        f"{pred_s:.4f} | {upd_s:.4f}"
    )

Time | Observation | Predicted H | Updated H | Predicted S | Updated S
   1 |       ALARM | 0.2000 | 0.6923 | 0.8000 | 0.3077
   2 |      NORMAL | 0.5846 | 0.1352 | 0.4154 | 0.8648
   3 |       ALARM | 0.1947 | 0.6851 | 0.8053 | 0.3149
   4 |       ALARM | 0.5796 | 0.9254 | 0.4204 | 0.0746
